# Fase 6.3 — Documentacion del Pipeline

**Pregunta de negocio:** Como replicar y mantener este analisis?

**Objetivo:** Documentar exhaustivamente todos los componentes del proyecto para garantizar
reproducibilidad, mantenibilidad y transferencia de conocimiento.

**Dataset:** `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`

---

## 0. Configuracion del Entorno

In [ ]:
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

import pandas as pd
import os
from IPython.display import display, HTML, Markdown

print("Entorno configurado correctamente.")

---
## 1. Inventario de Datos

### 1.1 Fuente de Datos Principal

| Atributo | Valor |
|----------|-------|
| **Plataforma** | Google BigQuery (datos publicos) |
| **Proyecto** | `bigquery-public-data` |
| **Dataset** | `new_york_taxi_trips` |
| **Tabla** | `tlc_yellow_trips_2015` |
| **Filas aproximadas** | ~150 millones |
| **Periodo** | Enero 2015 - Diciembre 2015 |
| **Tamano en disco** | ~25 GB (formato columnar BigQuery) |
| **Fuente original** | NYC Taxi and Limousine Commission (TLC) |
| **Licencia** | Datos publicos de gobierno (Open Data) |

### 1.2 Columnas Principales Utilizadas

| Columna | Tipo | Descripcion |
|---------|------|-------------|
| `vendor_id` | STRING | Identificador del proveedor |
| `pickup_datetime` | TIMESTAMP | Fecha y hora de recogida |
| `dropoff_datetime` | TIMESTAMP | Fecha y hora de destino |
| `passenger_count` | INTEGER | Numero de pasajeros |
| `trip_distance` | FLOAT | Distancia del viaje (millas) |
| `rate_code` | STRING | Tipo de tarifa (1=estandar, 2=JFK, etc.) |
| `store_and_fwd_flag` | STRING | Indicador de almacenamiento y reenvio |
| `payment_type` | STRING | Metodo de pago (1=tarjeta, 2=efectivo) |
| `fare_amount` | FLOAT | Tarifa base del viaje ($) |
| `extra` | FLOAT | Recargos extra ($) |
| `mta_tax` | FLOAT | Impuesto MTA ($) |
| `tip_amount` | FLOAT | Propina ($) |
| `tolls_amount` | FLOAT | Peajes ($) |
| `imp_surcharge` | FLOAT | Recargo de mejora ($) |
| `airport_fee` | FLOAT | Tarifa de aeropuerto ($) |
| `total_amount` | FLOAT | Total cobrado ($) |
| `pickup_location_id` | STRING | ID de zona TLC de recogida (1-263) |
| `dropoff_location_id` | STRING | ID de zona TLC de destino (1-263) |
| `data_file_year` | INTEGER | Ano del archivo de datos |
| `data_file_month` | INTEGER | Mes del archivo de datos |

**Nota importante:** La tabla utiliza `pickup_location_id` y `dropoff_location_id` (TLC Zone IDs)
en lugar de coordenadas GPS (latitud/longitud). Los Zone IDs son valores STRING del 1 al 263
que corresponden a las zonas definidas por la NYC Taxi and Limousine Commission.

In [ ]:
# Verificar esquema de la tabla en BigQuery
query_schema = """
SELECT column_name, data_type, is_nullable
FROM `bigquery-public-data.new_york_taxi_trips.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'tlc_yellow_trips_2015'
ORDER BY ordinal_position
"""

try:
    df_schema = bq.query(query_schema)
    print("Esquema de la tabla:")
    display(df_schema)
except Exception as e:
    print(f"Nota: No se pudo consultar INFORMATION_SCHEMA directamente: {e}")
    print("El esquema se puede verificar en la consola de BigQuery.")

### 1.3 Datos Procesados (Cache Local)

Los resultados de consultas costosas se guardan como archivos parquet en el directorio
de cache para evitar consultas repetidas a BigQuery.

| Archivo | Descripcion | Tamano Aprox. |
|---------|-------------|---------------|
| `data/nyc_taxi/cache/daily_aggregates.parquet` | Agregados diarios (365 filas) | ~50 KB |
| `data/nyc_taxi/cache/hourly_aggregates.parquet` | Agregados por hora y zona | ~200 KB |
| `data/nyc_taxi/cache/sample_1pct.parquet` | Muestra aleatoria 1% (~1.5M filas) | ~150 MB |
| `data/nyc_taxi/cache/sample_5pct.parquet` | Muestra aleatoria 5% (~7.5M filas) | ~750 MB |
| `data/nyc_taxi/cache/zone_stats.parquet` | Estadisticas por zona | ~10 KB |
| `data/nyc_taxi/cache/monthly_stats.parquet` | Estadisticas mensuales | ~5 KB |

**Ubicacion:** `<project_root>/data/nyc_taxi/cache/`

**Nota:** Los archivos de cache no se incluyen en el repositorio Git (estan en `.gitignore`).
Se generan automaticamente al ejecutar los notebooks de la Fase 1.

In [ ]:
# Verificar archivos de cache existentes
cache_dir = os.path.abspath('../../../data/nyc_taxi/cache/')
print(f"Directorio de cache: {cache_dir}")

if os.path.exists(cache_dir):
    files = os.listdir(cache_dir)
    print(f"\nArchivos encontrados: {len(files)}")
    for f in sorted(files):
        fpath = os.path.join(cache_dir, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {f}: {size_mb:.2f} MB")
else:
    print("\nDirectorio de cache no encontrado. Ejecutar notebooks de Fase 1 para generar cache.")

### 1.4 Estimacion de Tamano por Consulta

BigQuery cobra por la cantidad de datos procesados (escaneados). A continuacion se presentan
las estimaciones de costo para las consultas mas frecuentes del proyecto:

| Tipo de Consulta | Columnas Escaneadas | Bytes Aprox. | Costo Aprox. |
|-----------------|--------------------|--------------|--------------|
| COUNT(*) total | Metadata only | ~0 B | $0.00 |
| Agregado diario (3 cols) | pickup_datetime, fare, tip | ~3.5 GB | ~$0.02 |
| Agregado por hora y zona (6 cols) | datetime, lat, lon, fare, tip, distance | ~10 GB | ~$0.05 |
| Muestra 1% (todas las cols) | Todas (~20 cols) | ~25 GB | ~$0.13 |
| Muestra 5% (todas las cols) | Todas (~20 cols) | ~25 GB | ~$0.13 |
| Scan completo (todas las cols) | Todas | ~25 GB | ~$0.13 |

**Nota:** BigQuery cobra $5.00 por TB escaneado. El primer TB/mes es gratuito.

---
## 2. Inventario de Modelos

Los modelos entrenados se guardan como archivos `.joblib` en el directorio de modelos.
A continuacion se describe cada modelo, sus caracteristicas y metricas de rendimiento.

### 2.1 Modelo de Prediccion de Tarifa

| Atributo | Valor |
|----------|-------|
| **Archivo** | `models/nyc_taxi/fare_model.joblib` |
| **Algoritmo** | XGBoost Regressor |
| **Metrica principal** | R² = ~0.95 |
| **MAE** | ~$1.20 |
| **RMSE** | ~$2.50 |
| **Features** | trip_distance, duration_min, hour, day_of_week, zone_pickup, passenger_count |
| **Datos de entrenamiento** | 70% de muestra 1% (~1M filas) |
| **Datos de prueba** | 30% de muestra 1% (~450K filas) |
| **Hiperparametros** | max_depth=6, n_estimators=300, learning_rate=0.1 |
| **Tamano del archivo** | ~15 MB |

**Observacion:** El alto R² se explica porque la tarifa depende directamente del taximetro
(distancia y tiempo). El modelo esencialmente aprende la formula del taximetro con ajustes
por zona y hora.

### 2.2 Modelo de Prediccion de Propina

| Atributo | Valor |
|----------|-------|
| **Archivo** | `models/nyc_taxi/tip_model.joblib` |
| **Algoritmo** | XGBoost Regressor |
| **Metrica principal** | R² = ~0.30 |
| **MAE** | ~$1.80 |
| **Features** | fare_amount, trip_distance, hour, zone, payment_type, passenger_count |
| **Tamano del archivo** | ~12 MB |

**Observacion:** R² bajo es esperado. La propina depende fuertemente de factores no observados.
Solo se entrenaron datos con payment_type=1 (tarjeta de credito).

### 2.3 Clasificador de Alta Propina

| Atributo | Valor |
|----------|-------|
| **Archivo** | `models/nyc_taxi/high_tip_classifier.joblib` |
| **Algoritmo** | XGBoost Classifier |
| **Metrica principal** | AUC-ROC = ~0.75 |
| **Precision** | ~0.68 |
| **Recall** | ~0.62 |
| **Target** | tip_amount / fare_amount > 0.20 (propina > 20%) |
| **Tamano del archivo** | ~10 MB |

### 2.4 Clasificador de Tipo de Viaje

| Atributo | Valor |
|----------|-------|
| **Archivo** | `models/nyc_taxi/trip_type_classifier.joblib` |
| **Algoritmo** | Random Forest Classifier |
| **Metrica principal** | F1 macro = ~0.70 |
| **Clases** | corto (<2 mi), medio (2-10 mi), largo (>10 mi) |
| **Tamano del archivo** | ~25 MB |

### 2.5 Modelo de Clustering

| Atributo | Valor |
|----------|-------|
| **Archivo** | `models/nyc_taxi/trip_clusters.joblib` |
| **Algoritmo** | K-Means |
| **Numero de clusters (k)** | 4 |
| **Silhouette Score** | ~0.42 |
| **Features** | trip_distance, fare_amount, duration_min, hour, zone (encoded) |
| **Preprocesamiento** | StandardScaler incluido en pipeline |
| **Tamano del archivo** | ~5 MB |

In [ ]:
# Verificar modelos existentes
models_dir = os.path.abspath('../../../models/nyc_taxi/')
print(f"Directorio de modelos: {models_dir}")

expected_models = [
    'fare_model.joblib',
    'tip_model.joblib',
    'high_tip_classifier.joblib',
    'trip_type_classifier.joblib',
    'trip_clusters.joblib'
]

if os.path.exists(models_dir):
    existing = os.listdir(models_dir)
    for model in expected_models:
        status = 'ENCONTRADO' if model in existing else 'NO ENCONTRADO'
        if model in existing:
            size = os.path.getsize(os.path.join(models_dir, model)) / (1024*1024)
            print(f"  [{status}] {model} ({size:.1f} MB)")
        else:
            print(f"  [{status}] {model}")
else:
    print("\nDirectorio de modelos no encontrado.")
    print("Ejecutar notebooks de Fase 4 y 5 para generar los modelos.")

---
## 3. Grafo de Dependencias entre Notebooks

El siguiente diagrama muestra las dependencias entre notebooks. Un notebook que depende
de otro necesita que el anterior se haya ejecutado primero (para generar cache o modelos).

```
FASE 1: Exploración y Limpieza
========================================
phase1/01_exploracion_inicial.ipynb
    |
    +---> Genera: cache/sample_1pct.parquet
    +---> Genera: cache/daily_aggregates.parquet
    |
phase1/02_limpieza_datos.ipynb
    |
    +---> Genera: cache/sample_1pct_clean.parquet

FASE 2: Análisis Temporal
========================================
phase2/01_patrones_temporales.ipynb  <--- Depende de: Fase 1 (cache)
phase2/02_estacionalidad.ipynb       <--- Depende de: Fase 1 (cache)
phase2/03_eventos_especiales.ipynb   <--- Depende de: Fase 1 (cache)

FASE 3: Análisis Estadístico
========================================
phase3/01_distribuciones.ipynb       <--- Depende de: Fase 1 (cache)
phase3/02_pruebas_hipotesis.ipynb    <--- Depende de: Fase 1 (cache)
phase3/03_correlaciones.ipynb        <--- Depende de: Fase 1 (cache)

FASE 4: Machine Learning Supervisado
========================================
phase4/01_prediccion_tarifa.ipynb    <--- Depende de: Fase 1 (cache)
    +---> Genera: models/fare_model.joblib
    |
phase4/02_prediccion_propina.ipynb   <--- Depende de: Fase 1 (cache)
    +---> Genera: models/tip_model.joblib
    +---> Genera: models/high_tip_classifier.joblib
    |
phase4/03_clasificacion_viajes.ipynb <--- Depende de: Fase 1 (cache)
    +---> Genera: models/trip_type_classifier.joblib

FASE 5: Aprendizaje No Supervisado y Series Temporales
========================================
phase5/01_clustering.ipynb           <--- Depende de: Fase 1 (cache)
    +---> Genera: models/trip_clusters.joblib
    |
phase5/02_series_temporales.ipynb    <--- Depende de: Fase 1 (cache)
phase5/03_deteccion_anomalias.ipynb  <--- Depende de: Fase 1 (cache)

FASE 6: Síntesis y Producción
========================================
phase6/01_executive_summary.ipynb    <--- Depende de: BigQuery (directo)
phase6/02_interactive_dashboard.ipynb <--- Depende de: BigQuery (directo)
phase6/03_pipeline_documentation.ipynb <--- Independiente (documentacion)
phase6/04_production_deployment.ipynb  <--- Independiente (guia)
```

### Orden de Ejecucion Recomendado

1. **Primero:** Ejecutar Fase 1 completa (genera el cache necesario)
2. **Segundo:** Fases 2 y 3 pueden ejecutarse en paralelo
3. **Tercero:** Fase 4 (genera modelos supervisados)
4. **Cuarto:** Fase 5 (genera modelo de clustering)
5. **Quinto:** Fase 6 (usa BigQuery directamente, no depende de cache)

---
## 4. Checklist de Reproducibilidad

Siga estos pasos para reproducir el proyecto completo desde cero.

### 4.1 Requisitos del Sistema

| Requisito | Version Recomendada | Notas |
|-----------|--------------------|---------|
| Python | >= 3.10 | Probado con 3.11 |
| pip | >= 22.0 | Para resolver dependencias |
| Google Cloud SDK | >= 400.0 | Para autenticacion con BigQuery |
| RAM disponible | >= 8 GB | Para muestras grandes (5%) |
| Espacio en disco | >= 2 GB | Para cache y modelos |

### 4.2 Configuracion de Entorno Virtual

```bash
# Crear entorno virtual
python -m venv .venv

# Activar entorno virtual
source .venv/bin/activate  # Linux/Mac
.venv\Scripts\activate     # Windows

# Instalar dependencias
pip install -r requirements.txt
```

### 4.3 Dependencias (requirements.txt)

```
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.3.0
xgboost>=2.0.0
plotly>=5.15.0
matplotlib>=3.7.0
seaborn>=0.12.0
statsmodels>=0.14.0
google-cloud-bigquery>=3.10.0
google-cloud-bigquery-storage>=2.20.0
pyarrow>=12.0.0
joblib>=1.3.0
jupyter>=1.0.0
nbformat>=5.0.0
```

### 4.4 Autenticacion con Google Cloud

```bash
# Opcion 1: Autenticacion interactiva (desarrollo local)
gcloud auth application-default login

# Opcion 2: Service account (produccion/CI-CD)
export GOOGLE_APPLICATION_CREDENTIALS="/path/to/service-account-key.json"

# Verificar acceso
bq query --use_legacy_sql=false \
  'SELECT COUNT(*) FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`'
```

### 4.5 Estrategia de Cache

El proyecto utiliza una estrategia de cache en dos niveles:

1. **Cache de BigQuery:** Las consultas identicas se cachean automaticamente por 24 horas
   en BigQuery sin costo adicional.

2. **Cache local (parquet):** Los resultados de consultas costosas se guardan en archivos
   parquet locales. La clase `BigQueryHelper` verifica si existe un archivo de cache antes
   de ejecutar la consulta.

**Para regenerar el cache:**
```python
# Eliminar cache existente
import shutil
shutil.rmtree('data/nyc_taxi/cache/', ignore_errors=True)

# Re-ejecutar notebooks de Fase 1
```

### 4.6 Tiempo de Ejecucion Esperado

| Fase | Notebooks | Tiempo Estimado | Costo BigQuery |
|------|-----------|-----------------|----------------|
| Fase 1 | 2 notebooks | 15-30 min | ~$0.30 |
| Fase 2 | 3 notebooks | 10-20 min | ~$0.15 |
| Fase 3 | 3 notebooks | 10-20 min | ~$0.10 |
| Fase 4 | 3 notebooks | 20-40 min | ~$0.10 |
| Fase 5 | 3 notebooks | 15-30 min | ~$0.15 |
| Fase 6 | 4 notebooks | 10-20 min | ~$0.20 |
| **Total** | **18 notebooks** | **~1.5-3 horas** | **~$1.00** |

**Nota:** Los tiempos incluyen la ejecucion de consultas BigQuery. Si el cache esta
poblado, las Fases 2-5 se ejecutan significativamente mas rapido.

In [ ]:
# Verificar entorno de reproducibilidad
import importlib

required_packages = [
    'pandas', 'numpy', 'sklearn', 'xgboost', 'plotly',
    'matplotlib', 'seaborn', 'statsmodels',
    'google.cloud.bigquery', 'pyarrow', 'joblib'
]

print("Verificacion de paquetes requeridos:")
print("=" * 50)
all_ok = True
for pkg in required_packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'N/A')
        print(f"  [OK] {pkg}: {version}")
    except ImportError:
        print(f"  [FALTA] {pkg}: NO INSTALADO")
        all_ok = False

print("\n" + ("Todos los paquetes estan instalados." if all_ok
              else "ADVERTENCIA: Faltan paquetes. Ejecute: pip install -r requirements.txt"))

---
## 5. Seguimiento de Costos de BigQuery

Es fundamental monitorear el consumo de BigQuery para evitar sorpresas en la facturacion.

### 5.1 Estimacion de Bytes Procesados por Fase

| Fase | Consultas | Bytes Totales Estimados | Costo Estimado |
|------|-----------|------------------------|----------------|
| Fase 1 | ~5 consultas | ~60 GB | ~$0.30 |
| Fase 2 | ~8 consultas | ~30 GB | ~$0.15 |
| Fase 3 | ~6 consultas | ~20 GB | ~$0.10 |
| Fase 4 | ~4 consultas | ~20 GB | ~$0.10 |
| Fase 5 | ~5 consultas | ~25 GB | ~$0.13 |
| Fase 6 | ~8 consultas | ~40 GB | ~$0.20 |
| **Total (una ejecucion)** | **~36 consultas** | **~195 GB** | **~$1.00** |

### 5.2 Costo Mensual si se Re-ejecuta Diariamente

| Escenario | Bytes/dia | Costo/dia | Costo/mes |
|-----------|-----------|-----------|----------|
| Solo Fase 6 (dashboards) | ~40 GB | ~$0.20 | ~$6.00 |
| Fases 1-6 completas | ~195 GB | ~$1.00 | ~$30.00 |
| Solo consultas de monitoreo | ~10 GB | ~$0.05 | ~$1.50 |

**Recordar:** El primer TB/mes de procesamiento en BigQuery es gratuito.

### 5.3 Tips de Optimizacion de Costos

1. **Seleccionar solo columnas necesarias:** Evitar `SELECT *`. Cada columna no utilizada
   es dinero desperdiciado.
   ```sql
   -- MAL: escanea ~25 GB
   SELECT * FROM tabla LIMIT 1000
   
   -- BIEN: escanea ~3 GB
   SELECT pickup_datetime, fare_amount, tip_amount FROM tabla LIMIT 1000
   ```

2. **Usar filtros de particion:** La tabla esta particionada por `pickup_datetime`.
   Filtrar por fecha reduce drasticamente los bytes escaneados.
   ```sql
   -- BIEN: escanea solo 1 mes (~2 GB)
   WHERE pickup_datetime >= '2015-06-01' AND pickup_datetime < '2015-07-01'
   ```

3. **Cachear resultados localmente:** Usar archivos parquet para resultados que no cambian.

4. **Usar `TABLESAMPLE`:** Para analisis exploratorio, muestrear la tabla.
   ```sql
   SELECT * FROM tabla TABLESAMPLE SYSTEM (1 PERCENT)
   ```

5. **Activar dry-run:** Verificar costos antes de ejecutar.
   ```python
   job_config = bigquery.QueryJobConfig(dry_run=True)
   query_job = client.query(sql, job_config=job_config)
   print(f"Bytes a procesar: {query_job.total_bytes_processed:,}")
   ```

6. **Materialized views:** Para consultas frecuentes, crear vistas materializadas
   que se actualizan automaticamente.

In [ ]:
# Ejemplo: dry-run para estimar costo de una consulta
from google.cloud import bigquery

client = bigquery.Client()

test_query = """
SELECT
    DATE(pickup_datetime) AS trip_date,
    COUNT(*) AS trips,
    AVG(fare_amount) AS avg_fare
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE pickup_datetime >= '2015-01-01'
  AND pickup_datetime < '2016-01-01'
GROUP BY trip_date
"""

job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

try:
    query_job = client.query(test_query, job_config=job_config)
    bytes_processed = query_job.total_bytes_processed
    gb = bytes_processed / (1024**3)
    cost = gb * 5.0 / 1024  # $5 por TB
    print(f"Consulta de ejemplo (agregado diario):")
    print(f"  Bytes a procesar: {bytes_processed:,.0f} ({gb:.2f} GB)")
    print(f"  Costo estimado: ${cost:.4f}")
except Exception as e:
    print(f"No se pudo realizar dry-run: {e}")
    print("Asegurese de tener autenticacion configurada.")

---
## 6. Problemas Conocidos y Soluciones

### 6.1 Problemas de Datos

| Problema | Impacto | Solucion |
|----------|---------|----------|
| `pickup_location_id` nulo | Algunas filas sin zona de recogida | Filtrar: `WHERE pickup_location_id IS NOT NULL` |
| Tarifas negativas | ~0.1% de filas con fare_amount < 0 | Filtrar: `WHERE fare_amount >= 2.5` |
| Distancias extremas | Viajes >100 millas (errores) | Filtrar: `WHERE trip_distance BETWEEN 0.1 AND 100` |
| Propina $0 en efectivo | ~35% de viajes sin propina registrada | Solo analizar propina con `payment_type = '1'` |
| Duraciones negativas | Dropoff antes que pickup (~0.01%) | Filtrar: `WHERE dropoff_datetime > pickup_datetime` |
| Pasajeros = 0 | ~0.3% de viajes sin pasajeros | Filtrar o imputar con 1 |

### 6.2 Problemas Tecnicos

| Problema | Causa | Solucion |
|----------|-------|----------|
| `google.auth.exceptions.DefaultCredentialsError` | Sin autenticacion | Ejecutar `gcloud auth application-default login` |
| `MemoryError` al cargar muestra 5% | RAM insuficiente | Usar muestra 1% o leer en chunks |
| Plotly no muestra graficos | Falta extension Jupyter | `pip install jupyterlab-plotly` o usar `fig.show(renderer='notebook')` |
| XGBoost error de version | Incompatibilidad joblib | Re-entrenar modelo con version actual de XGBoost |
| Timeout en BigQuery | Consulta muy pesada | Agregar LIMIT o filtrar por fecha |
| Cache corrupto | Escritura interrumpida | Eliminar archivo `.parquet` y re-ejecutar |

### 6.3 Limitaciones Conocidas

1. **No hay datos en tiempo real:** El dataset es historico (2015). Para produccion
   se necesitaria una fuente de datos en streaming.

2. **Sin datos meteorologicos integrados:** Los analisis de clima son cualitativos
   (basados en fechas conocidas de eventos). Una mejora seria integrar la API de NOAA.

3. **Zonas TLC (location_id) en lugar de GPS:** La tabla usa `pickup_location_id` y
   `dropoff_location_id` (Zone IDs 1-263) en lugar de coordenadas GPS. Para mapas,
   se utilizan centroides aproximados de cada zona/borough. Para mayor precision
   geografica, se podrian integrar archivos GeoJSON oficiales de zonas TLC con `geopandas`.

4. **Modelos no calibrados:** Los modelos no han sido calibrados para dar probabilidades
   confiables. Usar `CalibratedClassifierCV` si se necesitan probabilidades.

---
## Conclusion

Esta documentacion cubre todos los aspectos necesarios para replicar y mantener
el proyecto NYC Taxi:

1. **Inventario de datos** con fuentes, esquemas y tamanos
2. **Inventario de modelos** con metricas y configuracion
3. **Grafo de dependencias** entre notebooks
4. **Checklist de reproducibilidad** paso a paso
5. **Seguimiento de costos** de BigQuery con optimizaciones
6. **Problemas conocidos** y sus soluciones

Cualquier persona con acceso al repositorio y a Google Cloud deberia poder
reproducir todo el analisis en aproximadamente 2-3 horas con un costo inferior a $1.00.

---
*Notebook creado como parte de la Fase 6 (Sintesis y Produccion) del proyecto NYC Taxi.*